# LieDetector YOLO11 重训 (新画面 + 隐身期强化)
数据: lie-detector-new-v1 (新画面采集, 遥测GT逐像素标注, 含大量 hidden 期 vortex 帧)
目标: 隐身期(纸纹漩涡形变)检出 + 可见期白团 + 背景图形

In [ ]:
!pip install -q ultralytics
from ultralytics import YOLO
import shutil, glob, os, yaml

In [ ]:
# 挂载数据集 (Kaggle 自动解压 zip -> dataset_yolo_new/; 保留 zip 兜底)
CAND = ["/kaggle/input/lie-detector-new-v1",
        "/kaggle/input/datasets/pancakesman/lie-detector-new-v1"]
DATASET_DIR = next(p for p in CAND if os.path.isdir(p))
print("dataset mount:", DATASET_DIR)

zips = glob.glob(f"{DATASET_DIR}/**/*.zip", recursive=True)
yamls = glob.glob(f"{DATASET_DIR}/**/data.yaml", recursive=True)
if zips:
    get_ipython().system(f'unzip -q "{zips[0]}" -d /kaggle/working/')
    work_yaml = "/kaggle/working/dataset_yolo_new/data.yaml"
    base = "/kaggle/working/dataset_yolo_new"
    with open(work_yaml) as f:
        cfg = yaml.safe_load(f)
else:
    src_yaml = yamls[0]
    base = os.path.dirname(src_yaml)
    with open(src_yaml) as f:
        cfg = yaml.safe_load(f)
    work_yaml = "/kaggle/working/data.yaml"
cfg["path"] = base
with open(work_yaml, "w") as f:
    yaml.safe_dump(cfg, f)
n_train = len(glob.glob(f"{base}/train/images/*"))
n_val = len(glob.glob(f"{base}/val/images/*"))
print(f"train {n_train} / val {n_val}")
print(cfg)

In [ ]:
model = YOLO("yolo11n.pt")
results = model.train(
    data=work_yaml,
    epochs=80,
    imgsz=640,
    batch=32,
    patience=30,
    project="/kaggle/working/runs",
    name="train",
)

In [ ]:
metrics = model.val()
print(f"mAP50-95: {metrics.box.map:.4f}, mAP50: {metrics.box.map50:.4f}")
names = metrics.names
for i, ci in enumerate(metrics.box.ap_class_index):
    print(f"{names[ci]:<12} mAP50={metrics.box.ap50[i]:.4f} mAP50-95={metrics.box.ap[i]:.4f}")
best = "/kaggle/working/runs/train/weights/best.pt"
shutil.copy(best, "/kaggle/working/best.pt")
print("done -> /kaggle/working/best.pt")